In [1]:
filename = '0_rect.js'

In [2]:
import subprocess
import textwrap

js_wrapper = textwrap.dedent("""
// file system
const fs = require('fs'); 
const code = fs.readFileSync('"""+filename+"""', 'utf8');

// mock object
const bot = { interrupt_code: false };
const world = {
  getPosition: () => ({ x: 0, y: 0, z: 0 }), // StartXYZ (bot standing pos)
  getBlockAtPosition: () => ({ name: 'air' }) // when bot asks block info
};
function log(){}

// placeBlock override
const skills = {
  breakBlockAt: async () => {},
  placeBlock: async (_bot, block, x, y, z) => {
    console.log(JSON.stringify({x, y, z, material: block}));
  }
};

(async () => {
  const f = eval(code);
  if (typeof f === 'function') await f(bot);
})();
""")

result = subprocess.run(
    ["node", "-e", js_wrapper],
    capture_output=True,
    text=True
)

# parsing
coords = []
for line in result.stdout.splitlines():
    try:
        coords.append(eval(line))
    except:
        pass

coords


[{'x': 0, 'y': 0, 'z': 0, 'material': 'stone'},
 {'x': 0, 'y': 0, 'z': 1, 'material': 'stone'},
 {'x': 0, 'y': 0, 'z': 2, 'material': 'stone'},
 {'x': 0, 'y': 0, 'z': 3, 'material': 'stone'},
 {'x': 0, 'y': 0, 'z': 4, 'material': 'stone'},
 {'x': 0, 'y': 0, 'z': 5, 'material': 'stone'},
 {'x': 0, 'y': 0, 'z': 6, 'material': 'stone'},
 {'x': 0, 'y': 0, 'z': 7, 'material': 'stone'},
 {'x': 0, 'y': 0, 'z': 8, 'material': 'stone'},
 {'x': 0, 'y': 0, 'z': 9, 'material': 'stone'},
 {'x': 0, 'y': 0, 'z': 10, 'material': 'stone'},
 {'x': 0, 'y': 0, 'z': 11, 'material': 'stone'},
 {'x': 0, 'y': 0, 'z': 12, 'material': 'stone'},
 {'x': 0, 'y': 0, 'z': 13, 'material': 'stone'},
 {'x': 0, 'y': 0, 'z': 14, 'material': 'stone'},
 {'x': 0, 'y': 0, 'z': 15, 'material': 'stone'},
 {'x': 0, 'y': 0, 'z': 16, 'material': 'stone'},
 {'x': 0, 'y': 0, 'z': 17, 'material': 'stone'},
 {'x': 0, 'y': 0, 'z': 18, 'material': 'stone'},
 {'x': 0, 'y': 0, 'z': 19, 'material': 'stone'},
 {'x': 1, 'y': 0, 'z': 0, 'mat

In [3]:
def _to_int(v): return int(str(v).strip())

def _xyz_list(items):
    xs, ys, zs = [], [], []
    for c in items:
        xs.append(_to_int(c["x"]))
        ys.append(_to_int(c["y"]))
        zs.append(_to_int(c["z"]))
    return xs, ys, zs

def eval_rect_with_material(items, expected_material="stone"):
    # 1) block cnt check
    if len(items) != 600:
        return False, f"Count mismatch: {len(items)} (expected 600)"

    # 2) size check
    xs, ys, zs = _xyz_list(items)
    dx = max(xs) - min(xs)
    dy = max(ys) - min(ys)
    dz = max(zs) - min(zs)
    size_ok = ((dx, dz) in {(14, 19), (19, 14)}) and (dy == 1)
    if not size_ok:
        return False, f"Size mismatch: dx={dx}, dy={dy}, dz={dz}"

    # 3) material check
    unique = {c.get("material") for c in items}
    if unique != {expected_material}:
        return False, f"Material mismatch: unique={sorted(unique)} (expected '{expected_material}')"

    return True, f"OK (count=600, dx={dx}, dy={dy}, dz={dz}, material='{expected_material}')"

# %% run
ok, msg = eval_rect_with_material(coords, expected_material="stone")
print(ok, msg)

True OK (count=600, dx=14, dy=1, dz=19, material='stone')
